### Adaptive Query Execution & Broadcast Join - Demo
**Adaptive Query Execution(AQE):** consiste en la reoptimización de consultas que tiene lugar durante su ejecución, basándose en estadísticas obtenidas en tiempo de ejecución.<br>
En Spark 3.0, AQE incluye tres funcionalidades principales:

- Dynamically coalescing shuffle partitions (Combinación dinámica de particiones Shuffle).
- Dynamically optimizing skew joins (Optimización dinámica de joins con asimetría de datos).
- Dynamically switching join strategies (Cambio dinámico de las estrategias de JOIN).

In [0]:
import pyspark.sql.functions as F
import time

In [0]:
df_transactions = spark.read.parquet("abfss://demo@databricksdeasac.dfs.core.windows.net/raw/data_skew/transactions.parquet/")
df_customers = spark.read.parquet("abfss://demo@databricksdeasac.dfs.core.windows.net/raw/data_skew/customers.parquet/")

In [0]:
df_transactions.printSchema()
df_transactions.show(5, False)

In [0]:
df_customers.printSchema()
df_customers.show(5, False)

#### Adaptive Query Execution(AQE)

In [0]:
spark.conf.set("spark.sql.adaptive.enabled", "true")

In [0]:
spark.conf.get("spark.sql.adaptive.enabled")

In [0]:
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

In [0]:
spark.conf.get("spark.sql.adaptive.skewJoin.enabled")

In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [0]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

In [0]:
# Mostrar los 5 clientes con mayor cantidad de transacciones únicas
(
    df_transactions
    .groupBy("cust_id")
    .agg(F.countDistinct("txn_id").alias("ct"))
    .orderBy(F.desc("ct"))
    .show(5, False)
)

In [0]:
# Capturar las transacciones con la información de los clientes
df_txn_details = (
    df_transactions.join(
        df_customers,
        on = "cust_id",
        how = "inner"
    )
)

In [0]:
# Contar los registros del JOIN y medir el tiempo que se tarda en completar.
start_time = time.time()
display(df_txn_details.count())
print("")
print(f"time_taken: {time.time() - start_time}")

#### Broadacast Join

In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)

In [0]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

In [0]:
# Capturar las transacciones con la información de los clientes
df_txn_details = (
    df_transactions.join(
        F.broadcast(df_customers),
        on = "cust_id",
        how = "inner"
    )
)

In [0]:
# Contar los registros del JOIN y medir el tiempo que se tarda en completar.
start_time = time.time()
df_txn_details.count()
print("")
print(f"time_taken: {time.time() - start_time}")